In [38]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanSquaredError
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

In [39]:
# Set random seed for reproducibility
tf.random.set_seed(42)

In [40]:
# Load the input and target data
X = pd.read_excel('data1319.xlsx', sheet_name=0, usecols='D:S', skiprows=1, nrows=765).values
y = pd.read_excel('data1319.xlsx', sheet_name=0, usecols='T', skiprows=1, nrows=765).values

In [41]:
# Expand dimensions of target data if necessary
t = np.expand_dims(y, axis=1) if y.ndim == 1 else y

In [42]:
# Normalize input and target data
input_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

In [43]:
x_scaled = input_scaler.fit_transform(X)
t_scaled = target_scaler.fit_transform(t)

In [44]:
# Split data into training, validation, and testing sets
x_train, x_temp, t_train, t_temp = train_test_split(x_scaled, t_scaled, test_size=0.3, random_state=42)
x_val, x_test, t_val, t_test = train_test_split(x_temp, t_temp, test_size=0.5, random_state=42)

In [45]:
# Create a fitting network
hidden_layer_size = 10
model = Sequential([
    Dense(hidden_layer_size, activation='relu', input_shape=(x_scaled.shape[1],)),
    Dense(t_scaled.shape[1], activation='relu')
])

/Users/charlesyamamura/ls/envs/aienv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [46]:
# Display the model summary
print("Model Summary:")
model.summary()

Model Summary:


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 10)             │           170 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 181 (724.00 B)

 Trainable params: 181 (724.00 B)

 Non-trainable params: 0 (0.00 B)

In [47]:
# Compile the model
model.compile(optimizer=Adam(), loss=MeanSquaredError(), metrics=['mae'])

In [48]:
# Train the network
history = model.fit(x_train, t_train, epochs=100, batch_size=10,
                    validation_data=(x_val, t_val), verbose=1)

Epoch 1/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.6704 - mae: 0.6796 - val_loss: 0.2241 - val_mae: 0.3760
Epoch 2/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.1280 - mae: 0.2897 - val_loss: 0.1350 - val_mae: 0.3027
Epoch 3/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.1075 - mae: 0.2562 - val_loss: 0.1270 - val_mae: 0.2934
Epoch 4/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.1002 - mae: 0.2457 - val_loss: 0.1180 - val_mae: 0.2797
Epoch 5/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0927 - mae: 0.2347 - val_loss: 0.1091 - val_mae: 0.2675
Epoch 6/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0856 - mae: 0.2239 - val_loss: 0.1006 - val_mae: 0.2561
Epoch 7/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0788 - mae: 0.2138 - val_loss: 0.0925 - val_mae: 0.2448
Epoch 8/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0726 - mae: 0.2045 - val_loss: 0.0851 - val_mae: 0.2337
Epoch 9/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.06

In [49]:
# Test the network
y_pred = model.predict(x_test)
performance = model.evaluate(x_test, t_test, verbose=0)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 


In [50]:
# Save the trained model
model_save_path = "mktRegMdl.keras"
model.save(model_save_path)
print(f"Model saved at: {model_save_path}")

Model saved at: mktRegMdl.keras


In [51]:
# Calculate training and testing adjusted accuracies
train_mse = model.evaluate(x_train, t_train, verbose=0)[0]
test_mse = performance[0]
target_variance = np.var(t_scaled)

In [52]:
train_accuracy = max(0, 1 - (train_mse / target_variance))
test_accuracy = max(0, 1 - (test_mse / target_variance))

In [53]:
# Calculate R-squared for training and testing
train_pred = model.predict(x_train)
test_pred = model.predict(x_test)

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [54]:
train_r2 = r2_score(t_train, train_pred)
test_r2 = r2_score(t_test, test_pred)

In [55]:
# Print performance results
print("\nPerformance Results:")
print("Test Performance (MSE):", test_mse)
print("Adjusted Training Accuracy:", train_accuracy)
print("Adjusted Testing Accuracy:", test_accuracy)
print("Training R-squared:", train_r2)
print("Testing R-squared:", test_r2)


Performance Results:
Test Performance (MSE): 0.012860553339123726
Adjusted Training Accuracy: 0.7944397808416177
Adjusted Testing Accuracy: 0.7492411466981841
Training R-squared: 0.7900474966825193
Testing R-squared: 0.7724169697232793


In [57]:
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(t_test, y_pred)
mae

0.08075864404729736

In [58]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(t_test, y_pred)
mse

0.012860554703092551

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(8, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# Plot adjusted accuracy results
epochs = range(1, len(history.history['loss']) + 1)
train_accuracy_epoch = [max(0, 1 - (loss / target_variance)) for loss in history.history['loss']]
val_accuracy_epoch = [max(0, 1 - (loss / target_variance)) for loss in history.history['val_loss']]
plt.figure(figsize=(8, 6))
plt.plot(epochs, train_accuracy_epoch, label='Training Adjusted Accuracy')
plt.plot(epochs, val_accuracy_epoch, label='Validation Adjusted Accuracy')
plt.legend()
plt.xlabel('Epochs')
plt.ylabel('Adjusted Accuracy')
plt.title('Training and Validation Adjusted Accuracy')
plt.show()

In [ ]:
# Optional: Generate a simple function for deployment
def my_neural_network_function(new_x):
    new_x_scaled = input_scaler.transform(new_x)
    y_scaled = model.predict(new_x_scaled)
    return target_scaler.inverse_transform(y_scaled)